In [ ]:
# Cell 1 - Clone and setup (siddhesh branch):
repo_url = "https://github.com/siddheshkadane01/Site-Reliability-Server/"
branch_name = "siddhesh"
!git clone --branch {branch_name} --single-branch {repo_url}
repo_dir = repo_url.rstrip("/").split("/")[-1]
%cd $repo_dir

In [ ]:
# Cell 2 - All installs in one cell, correct order:
%pip install -r requirements.txt
%pip install -r requirements_rl.txt

In [ ]:
# Cell 3 - Start FastAPI server:
import subprocess, time, requests
log_file = open("server.log", "w")
server_process = subprocess.Popen(
    ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "7860"],
    stdout=log_file, stderr=log_file
  )
time.sleep(8)
try:
    res = requests.get("http://localhost:7860/health", timeout=5)
    print("✅ Server running" if res.status_code == 200 
          else f"❌ Unexpected status: {res.status_code}")
except Exception as e:
    print("❌ Server failed:", e)
    print(open("server.log").read())

In [ ]:
# Cell 4 - Wandb login using secret:
import wandb, os
from google.colab import userdata
os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
wandb.login()

In [ ]:
# Cell 5 - Baseline eval -> Train -> After-training eval
import os, json
os.environ["WANDB_PROJECT"] = "openenv-enterprise-grpo"
os.environ["WANDB_RUN_NAME"] = "grpo-enterprise-42"
os.environ["WANDB_MODE"] = "online"

MODEL_NAME = "meta-llama/Meta-Llama-3-8B-Instruct"
OUTPUT_DIR = "./artifacts/grpo-enterprise-llama3"

# 1) Baseline (pre-training) evaluation on deterministic scenarios
!python evaluate_grpo_checkpoint.py \
    --model_name {MODEL_NAME} \
    --env_url http://localhost:7860 \
    --tasks easy medium hard expert enterprise \
    --scenarios_per_task 2 \
    --max_steps 25 \
    --max_new_tokens 96 \
    --load_in_4bit \
    --output_json eval_before.json

# 2) Train
!python train_grpo.py --epochs 20 --max_steps 25 \
    --num_generations 8 --lora_dropout 0.0 \
    --learning_rate 5e-6 --logging_steps 10 \
    --max_scenarios_per_task 10 \
    --wandb_mode online \
    --env_url http://localhost:7860 \
    --model_name {MODEL_NAME} \
    --output_dir {OUTPUT_DIR}

# 3) Compare base vs trained adapter in one report
!python evaluate_grpo_checkpoint.py \
    --model_name {MODEL_NAME} \
    --compare_adapter_path {OUTPUT_DIR} \
    --env_url http://localhost:7860 \
    --tasks easy medium hard expert enterprise \
    --scenarios_per_task 2 \
    --max_steps 25 \
    --max_new_tokens 96 \
    --load_in_4bit \
    --output_json eval_after_compare.json

report = json.load(open("eval_after_compare.json"))
print("\n=== AFTER TRAINING DELTA ===")
print(json.dumps(report.get("delta", {}), indent=2))